In [3]:
import pandas as pd
from pathlib import Path

nb_dir = Path.cwd()
project_root = nb_dir.parents[1]
results_dir = project_root / "experiments" / "results"

# 1. Load sktime and original ROCKET results for ECG200
df_sktime = pd.read_csv(results_dir / "ecg200_rocket_sktime.csv")
df_orig   = pd.read_csv(results_dir / "ecg200_rocket_original.csv")

df_sktime, df_orig

(  dataset  n_kernels  train_size  test_size  accuracy
 0  ECG200      10000         100        100      0.89,
   dataset  n_kernels  train_size  test_size  accuracy  fit_predict_time_sec  \
 0  ECG200      10000         100        100       0.9            203.991061   
 
    total_time_sec  random_state  
 0      204.081068            42  )

In [4]:
compare = pd.DataFrame([
    {
        "impl": "sktime",
        "accuracy": df_sktime.loc[0, "accuracy"],
        "fit_time_sec": df_sktime.loc[0, "fit_predict_time_sec"],
    },
    {
        "impl": "original",
        "accuracy": df_orig.loc[0, "accuracy"],
        "fit_time_sec": df_orig.loc[0, "fit_predict_time_sec"],
    },
])
compare

KeyError: 'fit_predict_time_sec'

### Comparison: sktime ROCKET vs original ROCKET on ECG200

On ECG200, both implementations achieve very similar accuracy: 0.89 for the sktime ROCKET baseline and 0.90 for our own original ROCKET implementation with 10,000 kernels, suggesting that the core ROCKET idea is reproduced correctly in terms of classification performance. At the same time, there is a large gap in runtime: the sktime implementation finishes in under a second, whereas the current original implementation takes over 200 seconds, highlighting that our straightforward NumPy-based code is much less optimized than the reference implementation, which uses low‑level optimizations (e.g., Numba and careful looping) to realise ROCKET’s promised speed. This indicates that, for the purposes of this project, we can already use our implementation to study accuracy and behaviour, but further engineering work will be needed if we want to approach the efficiency reported in the original paper.

---

Include both datasets - ECG200 and GunPoint



In [ ]:
import pandas as pd
from pathlib import Path

nb_dir = Path.cwd()
project_root = nb_dir.parents[1]
results_dir = project_root / "experiments" / "results"

datasets = ["ecg200", "gunpoint"]
rows = []

for name in datasets:
    sk = pd.read_csv(results_dir / f"{name}_rocket_sktime.csv")
    org = pd.read_csv(results_dir / f"{name}_rocket_original.csv")

    rows.append({
        "dataset": sk.loc[0, "dataset"],
        "impl": "sktime",
        "accuracy": sk.loc[0, "accuracy"],
        "fit_time_sec": sk.loc[0, "fit_predict_time_sec"],
    })
    rows.append({
        "dataset": org.loc[0, "dataset"],
        "impl": "original",
        "accuracy": org.loc[0, "accuracy"],
        "fit_time_sec": org.loc[0, "fit_predict_time_sec"],
    })

compare = pd.DataFrame(rows)
compare

,dataset,impl,accuracy,fit_time_sec
0,ECG200,sktime,0.890000,1.615330
1,ECG200,original,0.900000,203.991061
2,GunPoint,sktime,0.960000,1.160142
3,GunPoint,original,0.993333,310.947697


In [ ]:
import pandas as pd
from pathlib import Path

nb_dir = Path.cwd()
project_root = nb_dir.parents[1]          # AI4TS-ROCKET
results_dir = project_root / "experiments" / "results"

datasets = ["ecg200", "gunpoint", "italypowerdemand", "electricdevices"]

rows = []

for name in datasets:
    sk_path = results_dir / f"{name}_rocket_sktime.csv"
    org_path = results_dir / f"{name}_rocket_original.csv"

    sk = pd.read_csv(sk_path)
    org = pd.read_csv(org_path)

    rows.append({
        "dataset": sk.loc[0, "dataset"],
        "impl": "sktime",
        "n_kernels": sk.loc[0, "n_kernels"],
        "accuracy": sk.loc[0, "accuracy"],
        "fit_predict_time_sec": sk.loc[0, "fit_predict_time_sec"],
        "total_time_sec": sk.loc[0, "total_time_sec"],
    })
    rows.append({
        "dataset": org.loc[0, "dataset"],
        "impl": "original",
        "n_kernels": org.loc[0, "n_kernels"],
        "accuracy": org.loc[0, "accuracy"],
        "fit_predict_time_sec": org.loc[0, "fit_predict_time_sec"],
        "total_time_sec": org.loc[0, "total_time_sec"],
    })

comparison_df = pd.DataFrame(rows)
comparison_df = comparison_df[
    [
        "dataset",
        "impl",
        "n_kernels",
        "accuracy",
        "fit_predict_time_sec",
        "total_time_sec",
    ]
]
comparison_df

,dataset,impl,n_kernels,accuracy,fit_predict_time_sec,total_time_sec
0,ECG200,sktime,10000,0.890000,1.615330,1.629722
1,ECG200,original,10000,0.900000,203.991061,204.081068
2,GunPoint,sktime,10000,0.960000,1.160142,1.172276
3,GunPoint,original,10000,0.993333,310.947697,311.031773
4,ItalyPowerDemand,sktime,10000,0.947522,1.239877,1.281017
5,ItalyPowerDemand,original,10000,0.950437,296.302562,296.396254
6,ElectricDevices,sktime,10000,0.667358,109.936939,110.518015
7,ElectricDevices,original,2000,0.692647,3396.872223,3397.329666


## ---Comparison Results---
On these four datasets, sktime is consistently much faster, while the original implementation is equal or slightly better in accuracy:

ECG200: sktime 0.890 vs original 0.900 accuracy, but 1.63 s vs 204.08 s total time.

GunPoint: sktime 0.960 vs original 0.993 accuracy, 1.17 s vs 311.03 s.

ItalyPowerDemand: sktime 0.948 vs original 0.950 accuracy, 1.28 s vs 296.40 s.

ElectricDevices: sktime 0.667 (10 000 kernels, 110.52 s) vs original 0.693 (2 000 kernels, 3 397.33 s).

So overall, the original ROCKET is up to a few percentage points more accurate, but typically 100–300× slower, even when it uses fewer kernels, making sktime the clearly more practical choice when runtime matters.



| dataset           | impl      | n_kernels | accuracy | fit_predict_time_sec | total_time_sec |
|-------------------|-----------|-----------|----------|----------------------|----------------|
| ECG200            | sktime    | 10000     | 0.890000 | 1.615330             | 1.629722       |
| ECG200            | original  | 10000     | 0.900000 | 203.991061           | 204.081068     |
| GunPoint          | sktime    | 10000     | 0.960000 | 1.160142             | 1.172276       |
| GunPoint          | original  | 10000     | 0.993333 | 310.947697           | 311.031773     |
| ItalyPowerDemand  | sktime    | 10000     | 0.947522 | 1.239877             | 1.281017       |
| ItalyPowerDemand  | original  | 10000     | 0.950437 | 296.302562           | 296.396254     |
| ElectricDevices   | sktime    | 10000     | 0.667358 | 109.936939           | 110.518015     |
| ElectricDevices   | original  | 2000      | 0.692647 | 3396.872223          | 3397.329666    |

In [ ]:
import numpy as np

pivot = comparison_df.pivot_table(
    index="dataset",
    columns="impl",
    values=["accuracy", "total_time_sec"]
)

rel_df = pd.DataFrame({
    "dataset": pivot.index,
    "acc_diff_orig_minus_sktime": pivot["accuracy"]["original"] - pivot["accuracy"]["sktime"],
    "time_ratio_orig_div_sktime": pivot["total_time_sec"]["original"] / pivot["total_time_sec"]["sktime"],
    "log10_time_ratio": np.log10(
        pivot["total_time_sec"]["original"] / pivot["total_time_sec"]["sktime"]
    ),
})

rel_df

,dataset,acc_diff_orig_minus_sktime,time_ratio_orig_div_sktime,log10_time_ratio
dataset,,,,
ECG200,ECG200,0.010000,125.224457,2.097689
ElectricDevices,ElectricDevices,0.025289,30.740053,1.487705
GunPoint,GunPoint,0.033333,265.322985,2.423775
ItalyPowerDemand,ItalyPowerDemand,0.002915,231.375726,2.364318


We store two result files from our ROCKET experiments:

1. `rocket_sktime_vs_original_raw.csv`  
   This file contains the **raw per-run metrics** for each dataset and implementation.  
   Columns:
   - `dataset`: name of the UCR dataset (e.g. ECG200, GunPoint, ItalyPowerDemand, ElectricDevices). [conversation_history:0]
   - `impl`: implementation identifier (`sktime` or `original`). [conversation_history:0]
   - `n_kernels`: number of convolutional kernels used. [conversation_history:0]
   - `accuracy`: classification accuracy on the test split. [conversation_history:0]
   - `fit_predict_time_sec`: wall-clock time for fitting the classifier and predicting on the test set (seconds). [conversation_history:0]
   - `total_time_sec`: end-to-end runtime including any additional overhead (seconds). [conversation_history:0]

2. `rocket_sktime_vs_original_relative.csv`  
   This file contains **derived, comparative metrics** that summarise how the original implementation behaves *relative* to sktime for each dataset.  
   Columns:
   - `dataset`: name of the dataset. [conversation_history:0]
   - `acc_diff_orig_minus_sktime`: accuracy(original) − accuracy(sktime); positive values mean original is more accurate. [conversation_history:0]
   - `time_ratio_orig_div_sktime`: total_time_sec(original) ÷ total_time_sec(sktime); values > 1 indicate how many times slower original is. [conversation_history:0]
   - `log10_time_ratio`: base-10 logarithm of the time ratio, making it easier to interpret “orders of magnitude” differences in runtime. [conversation_history:0]

We use the *raw* CSV when we need exact measurements or want to recompute other statistics, and the *relative* CSV when we want to directly discuss trade-offs between accuracy and runtime in the results and discussion sections.

In [ ]:
from pathlib import Path

nb_dir = Path.cwd()

project_root = nb_dir.parents[1]

results_dir = project_root / "experiments" / "results"
results_dir.mkdir(parents=True, exist_ok=True)

comparison_df.to_csv(results_dir / "rocket_sktime_vs_original_raw.csv", index=False)
rel_df.to_csv(results_dir / "rocket_sktime_vs_original_relative.csv", index=False)

print("Saved to:", results_dir)

Saved to: /Volumes/MananSSD/Developer/AI4TS-Rocket/experiments/results
